In [5]:
from dotenv import load_dotenv
load_dotenv()

False

In [6]:
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.schema import Document


doc = Document(
    text=("""
    ### 第七条 事假
    1. 员工因私事必须本人处理的，可申请事假。
    2. 事假需提前申请并获直属主管批准，紧急情况可事后补办手续。
    3. 事假为无薪假，按日扣除相应工资。
    4. 每月事假原则上不超过3天，全年累计不超过15天，特殊情况需经人力资源部及公司领导审批。
    """
    ),
    metadata={"title": "Vacation Questions"}
)

# Token 切片

splitter = TokenTextSplitter(
    chunk_size=32,
    chunk_overlap=4,
    separator="\n"
)

nodes = splitter.get_nodes_from_documents([doc])

for node in nodes:
    print(node.text)
    print(node.metadata)

Metadata length (6) is close to chunk size (32). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
### 第七条 事假
    1. 员工因私事必
{'title': 'Vacation Questions'}
因私事必须本人处理的，可申请事假。
    2
{'title': 'Vacation Questions'}
2. 事假需提前申请并获直属主管批
{'title': 'Vacation Questions'}
主管批准，紧急情况可事后补办手续。
{'title': 'Vacation Questions'}
续。
    3. 事假为无薪假，按日扣
{'title': 'Vacation Questions'}
按日扣除相应工资。
    4. 每月事假原则上
{'title': 'Vacation Questions'}
原则上不超过3天，全年累计不超过15天，特殊情
{'title': 'Vacation Questions'}
特殊情况需经人力资源部及公司领导审批。
{'title': 'Vacation Questions'}


In [7]:
# 句子切片
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.node_parser import SentenceWindowNodeParser
sentence_splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)
# evaluate_splitter(sentence_splitter, documents, question, ground_truth, "Sentence")

In [8]:
# 句子窗口切片
sentence_window_splitter = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)
# 注意：句子窗口切片需要特殊的后处理器
# MetadataReplacementPostProcessor 应该从 postprocessor 模块导入，而不是 node_parser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

# 注意：下面的代码需要先创建 index，这里只是示例
# 实际使用时需要先有 documents 和 index
# documents = SimpleDirectoryReader("data").load_data()
# nodes = sentence_window_splitter.get_nodes_from_documents(documents)
# index = VectorStoreIndex(nodes)
#
# query_engine = index.as_query_engine(
#     similarity_top_k=5,
#     streaming=True,
#     node_postprocessors=[MetadataReplacementPostProcessor(target_metadata_key="window")]
# )
# evaluate_splitter(sentence_window_splitter, documents, question, ground_truth, "Sentence Window")

In [12]:
# 完整的句子窗口切片示例
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

# 1. 加载文档
documents = SimpleDirectoryReader("data").load_data()

# 2. 使用句子窗口切片器切分文档
nodes = sentence_window_splitter.get_nodes_from_documents(documents)

# 3. 创建索引
index = VectorStoreIndex(nodes)

# 4. 创建查询引擎，使用 MetadataReplacementPostProcessor
query_engine = index.as_query_engine(
    similarity_top_k=5,
    streaming=True,
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ]
)

# 5. 查询
response = query_engine.query("你的问题")
print(response)

print("✅ MetadataReplacementPostProcessor 导入成功！")
print("💡 使用说明：")
print("   - MetadataReplacementPostProcessor 用于句子窗口切片")
print("   - 它会用 'window' 元数据中的完整窗口文本替换检索到的节点文本")
print("   - 这样可以提供更多上下文，同时保持检索的精确性")

2026-01-31 18:19:21,490 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-01-31 18:19:21,491 - INFO - Retrying request to /chat/completions in 0.479570 seconds
2026-01-31 18:19:26,724 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-01-31 18:19:26,725 - INFO - Retrying request to /chat/completions in 0.849588 seconds
2026-01-31 18:19:28,877 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-01-31 18:19:28,879 - INFO - Retrying request to /chat/completions in 1.955858 seconds
2026-01-31 18:19:32,675 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# 语义切片
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core import Settings
from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels
Settings.embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V3,
    embed_batch_size=6,
    embed_input_length=8192
)
semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model
)
evaluate_splitter(semantic_splitter, documents, question, ground_truth, "Semantic")

In [ ]:
# markdown 切片
from llama_index.core.node_parser import MarkdownNodeParser
markdown_splitter = MarkdownNodeParser()
evaluate_splitter(markdown_splitter, documents, question, ground_truth, "Markdown")